<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex05-cnn-and-gnn/Ex05_05_report.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.
- Goodfellow, Bengio & Courville, *Deep Learning*, MIT Press 2016.  Ch. 10 for recurrence and the LSTM.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_05 · Notebook 05 — The Report

**Deep Learning for Engineering · Aalborg University · Part 1**

This notebook assembles the deliverable for Ex_05: a short markdown report in
which you choose an architecture and defend the choice.

## What is being marked

Two questions carry most of the marks, one from each half of lecture block L5.

> **L5.1.** On the six-bus network, the dense baseline was the more accurate
> model and the graph network was the one that survived a relabelling. **Which
> would you deploy, and what would have to be true about the deployment for that
> to be the right choice?**

> **L5.2.** Four architectures came within a factor of two of each other on the
> load-forecasting problem, and the differences between them were comparable to
> the differences between seeds. **What would have to change about the data for
> the ranking to become meaningful?**

Neither has a fixed correct answer. Both have a correct *shape* of answer: a
claim, the number that supports it, the spread that qualifies it, and the
condition under which it would stop being true. An answer with a number and no
qualification scores less than one with both, even when the number is the same.

Then the **four questions from L3.2 slide 2**, asked about the model you chose:

1. What is the input, precisely?
2. What is the loss — what single number was minimised?
3. Where did the data come from, and who paid for it?
4. What happens when it is wrong?

These four are how every exercise report in this course is marked, from Ex_03 to
Ex_12.

## How to use this notebook

Run notebooks 01, 03 and 04 first — this one loads the `.npz` files they wrote
and refuses to build a report without them. Then fill in the `ANSWERS`
dictionary, run the rest, and check the printed report before submitting it.

---

## 0 · Load what the other notebooks produced

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_5_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex05-cnn-and-gnn/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
import os
import numpy as np

import Ex_5_core as core

needed = {"nb01": "nb01_cnn.npz",
          "nb03": "nb03_gnn.npz",
          "nb04": "nb04_sequence.npz"}

results = {}
missing = []
for key, filename in needed.items():
    path = os.path.join(core.OUTPUT_DIR, filename)
    if os.path.exists(path):
        results[key] = np.load(path, allow_pickle=True)
        print(f"loaded {filename}")
    else:
        missing.append(filename)

if missing:
    raise FileNotFoundError(
        "missing " + ", ".join(missing)
        + " — run notebooks 01, 03 and 04 before this one")

---

## 0b · Your personal seed

Every notebook in this exercise set fixes the seed to 0, so the printed
"what you should see" blocks are true on any machine. That is right for
checking your work and wrong for reporting it — with one seed the whole cohort
produces identical numbers.

So the numbers below are **yours**. Put your study number in, run the cell, and
quote what it prints in your answers where the questions ask for it. Your
supervisor can regenerate exactly these numbers from your study number alone,
so they are worth getting right and pointless to invent.

In [ ]:
STUDENT_NUMBER = "20241234"        # <- your AAU study number

SEED = core.personal_seed(STUDENT_NUMBER)
print("study number :", STUDENT_NUMBER)
print("your seed    :", SEED)

# Your own batch of radiographs, from the same procedural process.
X_you, y_you = core.weld_images(n_per_class=20, seed=SEED)
MEANS_YOU = [float(X_you[y_you == k].mean()) for k in range(len(core.CLASS_NAMES))]

print()
print(f"  your batch : {X_you.shape}")
for name, m in zip(core.CLASS_NAMES, MEANS_YOU):
    print(f"    mean pixel value, {name:<6s} : {m:.5f}")

**What you should see.** Three `loaded ...` lines. If you get a
`FileNotFoundError`, go back and run the notebook it names, all the way to the
cell that saves.

---

## 1 · The evidence, gathered

Everything you measured, in one place. Read it before you write anything: the
report is meant to be an argument from these numbers, not an essay decorated with
them.

In [ ]:
nb1, nb3, nb4 = results["nb01"], results["nb03"], results["nb04"]

print("=" * 66)
print("NOTEBOOK 01 — convolution against a dense layer, on weld radiographs")
print("=" * 66)
print(core.error_table(
    [["convolutional", f"{int(nb1['n_cnn']):,}", f"{float(nb1['acc_train']):.3f}",
      f"{float(nb1['acc_test']):.3f}"],
     ["dense", f"{int(nb1['n_mlp']):,}", f"{float(nb1['acc_train_mlp']):.3f}",
      f"{float(nb1['acc_test_mlp']):.3f}"]],
    ["model", "parameters", "train accuracy", "held-out accuracy"]))
print(f"\nparameter ratio for one 16 x 16 layer: dense {int(nb1['dense_16']):,}"
      f"  vs conv {int(nb1['conv_16']):,}"
      f"  = {float(nb1['dense_16']) / float(nb1['conv_16']):,.0f} x")

print()
print("=" * 66)
print("NOTEBOOK 03 — node regression on the six-bus network")
print("=" * 66)
print(core.error_table(
    [["DC power flow", "0", f"{float(nb3['rmse_theta_dc']):.5f}", "not predicted"],
     ["graph network", f"{int(nb3['n_gnn']):,}",
      f"{float(nb3['rmse_theta_gnn']):.5f}", f"{float(nb3['rmse_volt_gnn']):.5f}"],
     ["dense network", f"{int(nb3['n_mlp']):,}",
      f"{float(nb3['rmse_theta_mlp']):.5f}", f"{float(nb3['rmse_volt_mlp']):.5f}"]],
    ["model", "parameters", "angle RMSE [rad]", "voltage RMSE [p.u.]"]))
print(f"\npermutation gap   graph {float(nb3['gap_gnn']):.2e} rad"
      f"   dense {float(nb3['gap_mlp']):.2e} rad")
print(f"after relabelling angle RMSE   graph {float(nb3['rmse_gnn_perm']):.5f}"
      f"   dense {float(nb3['rmse_mlp_perm']):.5f}")
print(f"after a line trip angle RMSE   graph {float(nb3['rmse_theta_gnn_trip']):.5f}"
      f"   dense {float(nb3['rmse_theta_mlp_trip']):.5f}")
print("\ndepth sweep:")
print(core.error_table([[int(d), f"{int(n):,}", f"{rt:.5f}", f"{rv:.5f}"]
                        for d, n, rt, rv in nb3["depth_results"]],
                       ["layers", "parameters", "angle RMSE", "voltage RMSE"]))

print()
print("=" * 66)
print("NOTEBOOK 04 — one-step load forecasting")
print("=" * 66)
print(core.error_table(
    [[str(n), f"{int(p):,}", f"{v:.6f}", f"{v / float(nb4['mse_persistence']):.2f}",
      f"{s:.1f}"]
     for n, p, v, s in zip(nb4["names"], nb4["params"], nb4["val"],
                           nb4["seconds"])]
    + [["persistence", "0", f"{float(nb4['mse_persistence']):.6f}", "1.00", "0.0"]],
    ["model", "parameters", "held-out MSE", "vs persistence", "seconds"]))
print(f"\nseed spread   perceptron {np.round(nb4['seeds_perceptron'], 6)}")
print(f"              attention  {np.round(nb4['seeds_attention'], 6)}")

**What you should see.** Three blocks of tables reproducing what you saw in
each notebook. Nothing new — but seeing them together is what makes the report
writable, because the argument runs across them.

One observation to take into the writing. In all three notebooks the more
constrained model was **not** the more accurate one on the test set it was trained
for, and was the better choice anyway in two of the three cases. Notebook 01 is
the exception: there the constrained model won on accuracy as well, because the
constraint — translation equivariance — was exactly true of the data. That
contrast is the whole subject of the L5.1 question.

---

## 2 · Your answers

Fill in the dictionary. Write in full sentences; the strings become the report.

Length guidance: two to four sentences per entry for the four questions, and a
paragraph each for the two marked questions. A worked example of the expected
standard is printed further down.

In [ ]:
ANSWERS = {
    # ---- the model you are defending ------------------------------------
    "model": "",           # e.g. "the three-layer graph network from notebook 03"

    # ---- L5.1: the deployment question ----------------------------------
    "l51": "",
    # Which of the six-bus models would you deploy, and what would have to be
    # true about the deployment for that to be the right choice? Use the
    # permutation numbers and the accuracy numbers, and say which you weighted
    # more heavily and why.

    # ---- L5.2: when does the architecture ranking mean anything? ---------
    "l52": "",
    # Four architectures within a factor of two, with seed-to-seed spread of the
    # same size. What would have to change about the data for the ranking to
    # become meaningful? Name a specific property of the data, not just "more".

    # ---- the four questions from L3.2 slide 2 ---------------------------
    "q1_input": "",        # what is the input, precisely? shapes and units
    "q2_loss": "",         # what single number was minimised, and in what units
    "q3_data": "",         # where did the data come from, and who paid for it
    "q4_wrong": "",        # what happens when it is wrong

    # ---- one thing you would do next ------------------------------------
    "next": "",            # one experiment, and what it would settle
}

blank = [k for k, v in ANSWERS.items() if not v.strip()]
print("filled in:", len(ANSWERS) - len(blank), "of", len(ANSWERS))
if blank:
    print("still empty:", ", ".join(blank))

**What you should see.** `filled in: 9 of 9` once you are done. Until then it
lists what is missing.

---

## 3 · A worked example, so the standard is visible rather than guessed

This is an answer to question 4 — *what happens when it is wrong* — for the
six-bus graph network. It is not the only good answer, and it is not about your
model. It is here to show the level of specificity that scores well.

> **What happens when it is wrong.** The model estimates bus angles to about
> 0.0043 radians and voltage magnitudes to about 0.0017 per unit on held-out
> cases drawn from the same operating envelope. A wrong estimate has two distinct
> consequences depending on where it is wrong. At a metered bus the error is
> visible: the operator can compare against the phasor measurement and discard
> the model. At an unmetered bus it is not, and an angle error of a few
> hundredths of a radian on a heavily loaded corridor is enough to misjudge the
> direction of a marginal flow and dispatch generation the wrong way. The failure
> is silent — nothing about the model's output announces it — which is why the
> estimate at an unmetered bus must be labelled as an inference and not as a
> reading, in every plot handed to somebody who did not run the code. The
> operating envelope is the second failure mode: the training injections came
> from a fixed range and the tripped-line experiment showed the error rising by more
> than a factor of twenty when the network changed. Anything outside that envelope
> is extrapolation, and the model gives no indication that it has left it.

Note what makes that answer work: it names the error in physical units, separates
detectable from undetectable failure, names the consequence to somebody outside
the room, and identifies the condition under which the numbers stop applying. An
answer of the form "the predictions would be inaccurate" scores near zero.

---

## 4 · Build the report

In [ ]:
lines = []
lines.append("# Ex_05 report — CNN and GNN\n")
lines.append("*Deep Learning for Engineering, Part 1. Model defended: "
             + (ANSWERS["model"] or "**not stated**") + ".*\n")

lines.append("\n## Results\n")
lines.append("### Convolution against a dense layer (notebook 01)\n")
lines.append(core.error_table(
    [["convolutional", f"{int(nb1['n_cnn']):,}", f"{float(nb1['acc_test']):.3f}"],
     ["dense", f"{int(nb1['n_mlp']):,}", f"{float(nb1['acc_test_mlp']):.3f}"]],
    ["model", "parameters", "held-out accuracy"]))

lines.append("\n### Node regression on six buses (notebook 03)\n")
lines.append(core.error_table(
    [["DC power flow", "0", f"{float(nb3['rmse_theta_dc']):.5f}", "-", "-"],
     ["graph network", f"{int(nb3['n_gnn']):,}",
      f"{float(nb3['rmse_theta_gnn']):.5f}",
      f"{float(nb3['rmse_volt_gnn']):.5f}",
      f"{float(nb3['gap_gnn']):.1e}"],
     ["dense network", f"{int(nb3['n_mlp']):,}",
      f"{float(nb3['rmse_theta_mlp']):.5f}",
      f"{float(nb3['rmse_volt_mlp']):.5f}",
      f"{float(nb3['gap_mlp']):.1e}"]],
    ["model", "parameters", "angle RMSE [rad]", "voltage RMSE [p.u.]",
     "permutation gap [rad]"]))

lines.append("\n### Load forecasting (notebook 04)\n")
lines.append(core.error_table(
    [[str(n), f"{int(p):,}", f"{v:.6f}"]
     for n, p, v in zip(nb4["names"], nb4["params"], nb4["val"])]
    + [["persistence", "0", f"{float(nb4['mse_persistence']):.6f}"]],
    ["model", "parameters", "held-out MSE"]))
lines.append("\nSeed spread over three seeds: perceptron "
             + np.array2string(np.round(nb4["seeds_perceptron"], 6))
             + ", attention "
             + np.array2string(np.round(nb4["seeds_attention"], 6)) + ".\n")

lines.append("\n## L5.1 — which model would you deploy?\n")
lines.append(ANSWERS["l51"] or "*not answered*")
lines.append("\n\n## L5.2 — when does the architecture ranking mean anything?\n")
lines.append(ANSWERS["l52"] or "*not answered*")

lines.append("\n\n## The four questions (L3.2 slide 2)\n")
for question, key in zip(core.four_questions(),
                         ["q1_input", "q2_loss", "q3_data", "q4_wrong"]):
    lines.append(f"\n**{question}**\n\n" + (ANSWERS[key] or "*not answered*") + "\n")

lines.append("\n## What I would do next\n")
lines.append(ANSWERS["next"] or "*not answered*")
lines.append("\n")

report = "\n".join(lines)

os.makedirs(core.OUTPUT_DIR, exist_ok=True)
report_path = os.path.join(core.OUTPUT_DIR, "Ex05_report.md")
with open(report_path, "w", encoding="utf-8") as fh:
    fh.write(report)

print("wrote", report_path, f"({len(report)} characters)")
print("\n" + "=" * 70 + "\n")
print(report)

### The report as a PDF

Moodle shows a PDF inline and a `.md` only as a download, so the cell below
converts the report you just wrote into a PDF (with the figure, if one was
saved) and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex05_report.md into Ex05_report.pdf, with any figure
# saved as Ex05_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md = open("Ex05_report.md", encoding="utf-8").read()
figs = sorted(glob.glob("Ex05_report*.png"))
if figs:
    imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
    i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
    md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

html = markdown.markdown(md, extensions=["fenced_code", "tables"])
css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                       f"<body>{html}</body></html>", base_url=".").write_pdf("Ex05_report.pdf")
print("written Ex05_report.pdf", f"with {len(figs)} figure(s)" if figs else "")
try:
    from google.colab import files
    files.download("Ex05_report.pdf")
except ImportError:
    pass


**What you should see.** `wrote .../Ex05_outputs/Ex05_report.md`, then the
report printed in full. Read it. If a section says *not answered*, it will say
that to the marker too.

---

## 5 · Optional extensions

None of these is required. Each is about half an hour and each settles a question
the notebooks deliberately left open.

**Hold the parameter count fixed in the depth sweep.** Notebook 03's sweep
changed depth and parameter count together, so part of the improvement from one
layer to three is ordinary capacity rather than reach. Shrink the hidden width as
the depth grows — the equal-budget experiment from Ex_04 notebook 04 — and see
whether the flattening at three layers survives.

**Train the graph network across topologies.** Notebook 03 showed that being able
to *accept* a new adjacency matrix is not enough to survive a line trip. Build a
training set that mixes the intact network with several single-line
contingencies, feeding the correct $\hat{A}$ with each case, and test on a
contingency held out from training. This is the "one model, many grids"
formulation of L12.1, and it is the experiment that would justify the
architecture properly.

**Use the susceptances as edge features.** The adjacency matrix threw them away.
Replace $\hat{A}$ with a susceptance-weighted normalised matrix and see what it
buys. L12.2 does this in Part 2 with line conductance and susceptance.

**Break the translation assumption in notebook 01.** Restrict the defects to the
upper-left quadrant of the plate and retrain both models. The convolution's
advantage should shrink, and the size of the shrinkage is a measurement of how
much the architectural prior was worth.

**Fix notebook 04's residual autocorrelation.** It was about 0.5, which means
predictable structure was left on the table. Try a longer window, or add the
first difference as a second input channel, and see whether the autocorrelation
falls and the MSE follows.

---

## 6 · What Ex_05 was for

Three architectures, one idea, and one habit.

**The idea** is L5.1's opening sentence: a learned local rule, applied
everywhere. A convolution applies it on a grid, message passing applies it on a
graph, a recurrent cell applies it along a line of time. In every case the
parameter count is set by the *rule* and not by the size of the thing it is
applied to, and in every case the sharing encodes an assumption about the
problem — that position does not matter, that node numbering does not matter,
that the dynamics do not change with the hour.

**The habit** is to ask what the architecture is claiming and then check the
claim against the data. Notebook 01's claim was true and the constrained model
won outright. Notebook 03's claim was true and the constrained model lost on
accuracy while winning the thing that mattered. Notebook 04's claim was not
supported by data of that size, and no amount of architecture fixed it.

**And the six buses.** When L12.1 says *"you met graph neural networks in Part
1"*, it means notebook 03. The same six buses come back there carrying a
physics-informed state estimator, with a measurement mask instead of an
`is_reference` feature and a power-flow residual instead of a supervised target.
The architecture will be familiar; the formulation will not.